# Machine Learning for Membrane Development

Welcome. This notebook is a **worked example** of machine learning (ML) on a table of **membrane** experiments: how a thin film was made, how well water and salt went through it, and how we might *suggest* a better recipe.

**Machine learning** here means: a computer finds patterns in past experiments so it can guess the result of a *new* recipe. It is not a physics simulator and it does not “understand” chemistry.

**You need:** basic Python (variables, functions, run cells **from top to bottom**).  
**You do not need:** a membrane course, or Bayesian optimization.

## Learning objectives

By the end you should be able to:

1. Tell **features** (inputs we know) from **targets** (numbers we want to predict).
2. Load a table and spot missing cells and duplicate rows.
3. Add extra chemistry columns (ion size, charge) *before* training.
4. Split data into **train** vs **test**, fit a tree model, and read a **parity plot**.
5. Read **SHAP** plots as *hypotheses*, not as proof of a mechanism.
6. Run a short **inverse design** search (Ax) for **continuous** targets only.

**Two kinds of prediction**

- **Regression:** the target is a number (permeability, rejection). This tutorial’s KPIs are regression.
- **Classification:** the target is a label (`pass` / `fail`). The examples include that; Ax search is **not** used for labels.

A high test score means “the model matched held-out rows,” **not** “we discovered the true physics.”

On **Google Colab**, run the bootstrap cell in the next section **before** Section 0.


### Prerequisites and setup (Google Colab)

**What you will do.** Run the next cell **once** so Colab has the full project, not just this `.ipynb` file.

**Repository (repo):** the folder that contains `config.yaml`, `src/`, and `data/`. Opening a notebook from GitHub does **not** download those files by itself.

**What the bootstrap cell does**

1. Clones (or updates) `ML-MEM-Review` under `/content/ML-MEM-Review`.
2. Installs the Python package with `pip install -e ".[all]"` — **core ML + SHAP + Ax**. The `-e` means “editable”: code in `src/` is what you import. First time can take **several minutes** (PyTorch).
3. Turns on inline plots. You do **not** need Google Drive.

**What to look for.** `Repo root: /content/ML-MEM-Review` and `Colab bootstrap done.`

**Common pitfall.** Skipping this cell → later: missing `config.yaml` or `No module named utl_...`. After a GitHub update, run bootstrap again.

**Local computer.** The cell mostly enables plots. In a terminal, from the repo root, once:

```bash
pip install -e ".[all]"
```

Use the **same** Python as the notebook kernel. Full notes: [`docs/COLAB.md`](../../docs/COLAB.md).


In [ ]:
# @title Bootstrap (clone + install)
# === COLAB BOOTSTRAP (safe locally) =========================================
# Clones Oikiy/ML-MEM-Review if you only opened the .ipynb from GitHub,
# installs the package editable, and cd's to the repo root.
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
REPO_REF = os.environ.get("ML_MEM_REPO_REF", "main")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))
EXTRAS = 'all'

def _git_head(repo: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return ""

def _truthy(name: str) -> bool:
    return os.environ.get(name, "").strip().lower() in {"1", "true", "yes"}

if IN_COLAB:
    if COLAB_ROOT.exists() and (COLAB_ROOT / ".git").is_dir():
        if _truthy("ML_MEM_SKIP_PULL"):
            print(f"Skipping git update at {COLAB_ROOT} (ML_MEM_SKIP_PULL=1)")
        else:
            print(f"Updating clone at {COLAB_ROOT} to origin/{REPO_REF}")
            _run(["git", "-C", str(COLAB_ROOT), "fetch", "--depth", "1", "origin", REPO_REF])
            _run(["git", "-C", str(COLAB_ROOT), "reset", "--hard", "FETCH_HEAD"])
    elif not COLAB_ROOT.exists():
        _run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)])
    os.chdir(COLAB_ROOT)
    root = _find_repo_root(Path.cwd())
else:
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate the ML-MEM-Review repo (need config.yaml + src/). "
        "On Colab: re-run this cell or set ML_MEM_REPO_URL to your fork. "
        "Locally: open the notebook from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}  git={_git_head(root)[:12]}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = _truthy("ML_MEM_FORCE_INSTALL")
    head = _git_head(root)
    extras = EXTRAS
    pip_spec = str(root) if not extras else f"{root}[{extras}]"
    install_key = f"{head}\n{extras}"
    prev = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""
    if force or prev != install_key:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", pip_spec])
        try:
            marker.write_text(install_key + "\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (clone SHA + extras unchanged). Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on sys.path (no auto pip install).")

if EXTRAS:
    try:
        import ax  # noqa: F401
    except ImportError:
        pip_spec = f"{root}[{EXTRAS}]"
        print(f"Ax not found in this kernel; installing {pip_spec}")
        _run([sys.executable, "-m", "pip", "install", "-e", pip_spec])

print("Next: run the Section 0 import cell below.")
# === end bootstrap ===========================================================


## Section 0. Setup

**What you will do.** Load **`config.yaml`** — the control panel for this whole notebook — and set a random seed so splits are repeatable.

**What `config.yaml` is.** A text file of settings (paths, which column is the target, which model, how Ax searches). Later cells **read this file**; they should not hard-code the Excel name. To reuse the package on *your* table, you edit YAML, not this Python.

```yaml
data:
  dataset: "TutorialDataSet.xlsx"   # file inside data/raw/
  targets:                          # columns to predict
    - "Pure_Water_Permeability_..."
    - "solute1_Rejection"
```

**Words**

- **Random seed:** an integer that makes “random” shuffles the same next time (here from `modeling.random_state`).
- **Kernel:** the Python process behind the notebook. Installs must go into *this* kernel.

This notebook calls the **same functions** as:

```bash
python -m run_pipeline --config config.yaml
```

**What to look for.** Printed paths to the dataset, ion lookup tables, `models/`, and `outputs/figures/`. On Colab they sit under `/content/ML-MEM-Review/...`.

**Common pitfall.** Running this cell before bootstrap on Colab. A warning that Ax is missing is OK until Section 6.

**Try it yourself.** Does the Excel path come from a variable you typed in Python, or from `config.yaml`?


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
except Exception as exc:
    Client = None
    RangeParameterConfig = None
    ChoiceParameterConfig = None
    warnings.warn(
        f"Ax import skipped during setup: {exc}. "
        "You can still run Sections 1-5; install pip install -e '.[opt]' before Section 6.",
        RuntimeWarning,
    )

PROJECT_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(PROJECT_ROOT, "config.yaml")):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not find config.yaml above the current folder. "
            "Launch the notebook from inside the repository."
        )
    PROJECT_ROOT = parent
CODE_PATH = os.path.join(PROJECT_ROOT, "src")
if CODE_PATH not in sys.path:
    sys.path.insert(0, CODE_PATH)

from utl_config_loader import (
    get_data_paths,
    get_figures_dir,
    get_model_dir,
    load_config,
    resolve_targets,
)

config = load_config(project_root=PROJECT_ROOT)
print("Configuration loaded from:", config["_config_path"])
print("Same engine as: python -m run_pipeline --config config.yaml")

plotting_cfg = config["plotting"]
FONT_SIZE = plotting_cfg["font_size"]
FIG_SIZE = tuple(plotting_cfg["global_figure_size"])
DPI = plotting_cfg["dpi"]
plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["figure.figsize"] = FIG_SIZE
plt.rcParams["savefig.dpi"] = DPI

RANDOM_STATE = config["modeling"]["random_state"]
np.random.seed(RANDOM_STATE)

data_paths = get_data_paths(config)
DATA_PATH = data_paths["dataset"]
lookup_by_align_column = data_paths.get("lookup_by_align_column", {})
CATION_PROP_PATH = lookup_by_align_column.get("S1_Cation")
ANION_PROP_PATH = lookup_by_align_column.get("S1_Anion")
if CATION_PROP_PATH is None or ANION_PROP_PATH is None:
    raise KeyError(
        "config.yaml must define feature_engineering.lookups for S1_Cation and S1_Anion."
    )

MODEL_PATH = get_model_dir(config)
FIGURES_DIR = get_figures_dir(config)
os.makedirs(FIGURES_DIR, exist_ok=True)
targets = resolve_targets(config)

print("Dataset:", os.path.basename(DATA_PATH))
print("Targets:", targets)
print("Cation lookup:", os.path.basename(CATION_PROP_PATH))
print("Anion lookup:", os.path.basename(ANION_PROP_PATH))
print("Models:", MODEL_PATH)
print("Figures:", FIGURES_DIR)


## Section 1. Raw dataset — load and look

**What you will do.** Read the spreadsheet into a **DataFrame** (a table `df` with named columns) and look at its size and names.

**One row = one published experiment**, not a random sample from nature. Many rows can come from the same paper. That matters later: the model can look better than it is if the same recipe is copied twice.

**Features vs targets**

| Role | Meaning | Examples in this table |
|------|---------|------------------------|
| **Feature** (input) | What we *know* before predicting | monomer name, mass, polymerization time, `S1_Cation` |
| **Target** (output) | The number we want to **predict** | water permeability, solute rejection |

A **KPI** (key performance indicator) is just an important target: here, how fast water goes through and how well salt is rejected.

The bundled table is on the order of **~278 rows × ~36 columns**. If you pointed YAML at another file, your numbers will differ.

**What to look for.** `df.head()`: columns such as `DOI`, `Dope1_Monomer1_Name`, `solute1_Rejection`. Long names with `{Lm^{-2}...}` are units baked into the column name — awkward but unique.

**Try it yourself.** From the column list, write down **one feature** and **one target** before you continue.


In [ ]:
# Load the dataset into a pandas DataFrame
df = pd.read_excel(DATA_PATH)

# Show the shape and first few rows of the dataset
print("Dataset shape (rows, columns):", df.shape)
df.head()

### How the columns are grouped

Think of three notebooks stapled together:

1. **How the film was made** — monomers, additives, solvents, time, temperature (`Dope1_...`, `Dope2_...`).
2. **How it was tested** — pressure, feed, compaction *when the paper reported them*.
3. **What was in the water** — solute name and ions (`Solute1_Name`, `S1_Cation`, `S1_Anion`).

Section 2 will attach **numbers** (size, charge) to those ion *names*. You are still only **looking**, not cleaning.


### Missing values and duplicate rows

**What you will do.** Count empty cells and exact duplicate rows.

**Missing value:** a blank in the spreadsheet (pandas shows it as `NaN`, “not a number”). Too many blanks on a column you hoped to use is a warning, not an automatic delete.

**Duplicate row:** the same experiment copied twice. That can **inflate** scores because the model sees the same recipe in train and, unluckily, in test.

**What we will not do here.** We will not fill blanks yet. Filling (called **imputation**) happens **inside training**, using **only the train set**, so the test set stays honest. That avoids **leakage** (accidentally using test information while learning).

**What to look for.** `df.info()`: non-null counts. A printed duplicate count.

**Common pitfall.** Dropping every row that has *any* blank. On a small materials table you can throw away most of the data.

**Try it yourself.** If polymerization time is missing in 80% of rows, would you impute it, drop the column, or keep an “is missing?” flag? There is no single right answer — that is the point.


In [ ]:
# Check the data types and missing values
print("\nData types and missing values:")
print("---"*30)
print(df.info())

# Check for duplicates
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {num_duplicates}")

### Optional deeper dive: why values are missing

Skip this box if you want. Imputation is **not** applied in the next code cell.

Statisticians name three patterns:

- **MCAR** — missing by accident (a logger dropped a point). Simple averages are often OK if few cells are empty.
- **MAR** — missingness depends on **other columns we did record** (e.g. a method never reported when the polymer was cheap).
- **MNAR** — missing *because of the missing value itself* (extreme conditions never published). Blind averages can mislead.

Membrane papers also have **structured** gaps: a measurement was never in the protocol. Then dropping the column is more honest than inventing a mean.

**In this tutorial.** Section 4’s pipeline fills holes using **training data only**, then applies the same rule to test rows.


In [ ]:
# Implement duplicate-row handling identified above.
num_duplicates = df.duplicated().sum()
if num_duplicates > 0:
    print(f"Dropping {num_duplicates} duplicate rows (keeping the first occurrence).")
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    print(f"Rows remaining after deduplication: {len(df)}")
else:
    print("No duplicate rows to drop.")


In [ ]:
from utl_config_loader import resolve_features_to_model, resolve_targets

targets = resolve_targets(config)
print("data.targets in config.yaml:")
for name in targets:
    print(" -", name)
    if name not in df.columns:
        raise ValueError(f"Target {name!r} is not a column in the loaded table.")

print("\nfeatures.include lists model inputs. Ion-size columns (S1_cat_*, S1_an_*)")
print("are created in Section 2; they are not on the raw table yet.")
print("Configured feature names:")
for name in config.get("features", {}).get("include") or []:
    print(" -", name)


## Section 2. Feature engineering

**What you will do.** Call `feature_engineering` to **merge lookup tables** of ion properties onto the main table.

**Feature engineering** means: create extra input columns that might help the model, *before* fitting. Here we do not invent physics — we **join** known ion size and charge onto `S1_Cation` / `S1_Anion`.

A name like `Na+` is only text. Transport cares about **size** and **charge**. The lookup files (`cation_prop.xlsx`, `anion_prop.xlsx`) are small dictionaries: ion name → numbers. After the merge you should see columns such as Stokes radius.

```yaml
feature_engineering:
  lookups:
    - align_column: "S1_Cation"    # column in the main table
      lookup_key: "Cation"         # column in the ion file
      lookup_path: "data/raw/cation_prop.xlsx"
```

This is **not** training and **not** a simulator.

**What to look for.** `df_enriched` with extra `S1_cat_*` / `S1_an_*` columns. Targets unchanged. Printed “model features after enrichment” should include those ion columns (they are listed in `features.include` but do not exist on the *raw* table).

**Common pitfall.** Thinking the model is trained now. Ion names that do not match the lookup stay without those extra columns.

**Try it yourself.** After the cell runs, can you find a column that was **not** in `df.head()` in Section 1?


In [ ]:
from utl_feateng import feature_engineering
from utl_config_loader import resolve_features_to_model

df_enriched = feature_engineering(df, config=config, verbose=True)
features = resolve_features_to_model(config, df_enriched)
print(f"\n{len(features)} model features after enrichment (from config.yaml):")
for name in features:
    print(" -", name)
df_enriched


### The enriched table

`df_enriched` = original recipe columns **plus** ion descriptors.

This tutorial only adds a **small** ion set (enough to show the idea). You could later add solvent or polymer properties the same way in YAML.

**What to look for.** For solutes you care about, `S1_Cation` / `S1_Anion` produced real numbers, not blanks.


### Optional: columns that never change

**Variance** is “how much a column wiggles.” If every row has the same polymerization temperature, that column cannot help a prediction, but after **one-hot encoding** (turning names into 0/1 columns, Section 4) it can still add clutter.

The next cell **reports** near-constant columns. Modeling still uses `features.include` from YAML (same as `run_pipeline`), not a second secret filter.

**What to look for.** A printed list. Empty list = nothing was constant enough to flag.


In [ ]:
from utl_feateng import remove_low_variance_columns

variance_threshold = config.get("features", {}).get("low_variance_threshold", 1e-9)
_, dropped = remove_low_variance_columns(
    df_enriched,
    exclude_columns=targets,
    variance_threshold=variance_threshold,
)
print("Variance threshold:", variance_threshold)
print("Near-constant columns (not dropped for modeling; train_model uses config features):", dropped)


### Which knobs we keep

`features.include` in `config.yaml` is the allow-list of **model inputs** after enrichment: monomer identity, masses, time, solute, ion size/charge.

Section 6 will only **search** a subset of those (the ones you can change in the lab). Everything else the model still needs is held **constant** during search.

**Next.** Plots (quality control), then training. Both use `df_enriched` and the same YAML as `python -m run_pipeline`.


## Section 3. Exploratory plots (before training)

**What you will do.** Look at how the **targets** are spread, then a **correlation heatmap** of numeric columns.

**Why plot before training?** If almost all permeability values sit in a tiny band, a “good” score can be cheap (the model always guesses the average). If two numeric columns are nearly copies, the model may pick an arbitrary one.

These plots are **quality control**, not the scientific conclusion.

**Pearson correlation** is a number from **−1 to 1** for a *straight-line* relationship between two **numeric** columns:

- near 1: they rise together; near −1: one up, the other down; near 0: no linear link.
- It **ignores categories** (monomer *names*). A tree model can still use those names.

The heatmap **hides** pairs whose |correlation| is below a threshold from YAML (`data_analysis.correlation_thresholds`, here 0.5) so the picture is less noisy.

**What to look for.** `target_distributions.png` (spread, skew, outliers) and `pearson_heatmap.png`. On Colab they also appear under `outputs/figures/`.

**Common pitfall.** Dropping a chemically important knob only because Pearson is small. Nonlinear effects can still matter.

**Try it yourself.** Will monomer *name* appear in the Pearson heatmap? (It should not.)


In [ ]:
from utl_eda import plot_target_distributions, compute_and_plot_pearson_heatmap
from utl_fileIO import file_io

fig, _ = plot_target_distributions(df_enriched, target_columns=targets, config=config)
file_io("save", "figure", FIGURES_DIR, "target_distributions.png", data=fig)
print("Saved target_distributions.png")

numeric_df = df_enriched.select_dtypes(include=["number"])
threshold = (config.get("data_analysis", {}) or {}).get("correlation_thresholds", [0.5])[-1]
_, heatmap_fig = compute_and_plot_pearson_heatmap(
    numeric_df, threshold=threshold, config=config
)
file_io("save", "figure", FIGURES_DIR, "pearson_heatmap.png", data=heatmap_fig)
print("Saved pearson_heatmap.png")


### Reading the heatmap

Strong correlation in a **literature** table can mean “papers tend to report these two settings together,” not “A causes B.” **Correlation is not causation.**

Weak Pearson does not mean “useless.” That is why Section 4 uses **trees**, which can capture *combinations* (monomer × time × additive) that a straight line misses.

If two numerics are almost the same measurement, keep the one you can set in the lab — that choice also matters for the Section 6 search space.


## Section 4. Train a model (ExtraTrees)

**What you will do.** For each `data.targets` column, fit a model and draw a **parity plot** (predicted vs measured).

**Train vs test.** We split rows at random:

- **Train set** (~80%): the model may *learn* from these.
- **Test set** (~20%): held back. If it also does well here, it is not only memorizing homework.

If train looks great and test looks poor, that is **overfitting** — like practicing only the answer key and failing a new exam.

**ExtraTrees** is a crowd of **decision trees**. Each tree asks yes/no questions (“is time > 5 min?”). Averaging many random trees soaks up noise from individual papers. You do not need the math; treat it as a flexible curve-fitter for tables.

**One-hot encoding:** monomer *names* become 0/1 columns (PIP=1, PEI=0, …) so the trees can use categories. That happens **inside** the sklearn pipeline, using the train set only.

**Scores you will see**

- **R²** (often 0–1, can be negative): 1 = perfect; 0 = no better than predicting the average. Compare **train R²** and **test R²**.
- **RMSE / MAE**: typical error in the **same units as the KPI**. Smaller is better. RMSE punishes big misses more than MAE.

```yaml
modeling:
  model_selection: extratrees   # or randomforest, ridge, ...
  task: regression
  train_test_ratio: 0.2
```

Change `model_selection` in YAML, not in this cell. `force_retrain=True` so YAML changes are picked up.

**What to look for.** Train *and* test metrics; `parity_<target>.png` (near the diagonal = good); pickles in `models/`.

**Common pitfall.** Trusting train R² alone. A long target name in the figure title is normal.

**Try it yourself.** Train R² = 0.99 and test R² = 0.2 — would you trust Section 6’s recipe suggestions? Why or why not?


In [ ]:
from utl_fileIO import file_io
from utl_modelling import _safe_target, evaluate_and_plot_results, train_model

for target in targets:
    print(f"\n=== Modeling target: {target} ===")
    pipe, X_train, X_test, y_train, y_test = train_model(
        df=df_enriched,
        target=target,
        config_path=config["_config_path"],
        force_retrain=True,
    )
    fig, resolved_task = evaluate_and_plot_results(
        target=target,
        y_train=y_train,
        y_pred_train=pipe.predict(X_train),
        y_test=y_test,
        y_pred_test=pipe.predict(X_test),
        config=config,
        fig_size=FIG_SIZE,
    )
    plot_name = (
        f"confusion_{_safe_target(target)}.png"
        if resolved_task == "classification"
        else f"parity_{_safe_target(target)}.png"
    )
    file_io("save", "figure", FIGURES_DIR, plot_name, data=fig)
    print("Saved", plot_name)


### How to read a parity plot

Each point is one experiment: **measured** KPI on one axis, **predicted** on the other.

The **diagonal** (dashed line) means prediction = measurement. Points **above** the line: the model over-predicted; **below**: under-predicted.

**What to look for**

- A cloud hugging the diagonal **across the whole range**, not only in the middle.
- Train tight, test scattered → overfitting.
- A bend at high permeability → the model is weak there; inverse-design suggestions in that region are shaky.

Classification problems use a **confusion matrix** (counts of predicted vs true labels) instead. This tutorial’s KPIs are numbers, so we use parity.

**Try it yourself.** Where on the plot would “always under-predicts high rejection” show up?


## Section 5. SHAP — who is the model listening to?

**What you will do.** Load the **already saved** models and draw SHAP plots.

A test score says *whether* predictions match. **SHAP** (SHapley Additive exPlanations) says *which input columns* pushed a prediction up or down **in this fitted model**. That is an explanation of the **surrogate**, not a proof of membrane physics, and **not causation**.

You do not need the game-theory math. Read the pictures:

- Features at the **top** matter more on average.
- Color = high vs low feature value; left/right = decrease vs increase in the prediction.

Needs the `shap` package (`pip install -e ".[shap]"`). If it is missing, this cell tries to install it, or skips so you can still run Section 6.

**What to look for.** `shap_summary_*.png` and `shap_importance_*.png` in `outputs/figures/`. Only the **top 20** encoded features are drawn if there are many (one-hot names get long).

**Common pitfall.** “SHAP says additive loading *causes* rejection.” Safer: “In this model, higher additive loading was *associated with* higher/lower predicted rejection.”

**Try it yourself.** Circle one feature you could actually change in a lab. That is a candidate for the Ax **search space**. Features you cannot control belong in **constants**.


### Beeswarm vs bar

- **Beeswarm (summary):** each dot is one sample. Use this for **direction** (does high time push rejection up or down?).
- **Bar (importance):** mean |SHAP| — a simple **ranking** when you do not need direction.

Encoded names like `Dope1_Monomer1_Name_PIP` are the one-hot columns from Section 4. That is expected.


In [ ]:
try:
    import shap  # noqa: F401
except ImportError:
    import subprocess
    spec = f"{PROJECT_ROOT}[shap]"
    print("SHAP not found; installing", spec)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    import shap  # noqa: F401

from utl_config_loader import resolve_features_to_model
from utl_fileIO import file_io
from utl_modelling import _safe_target
from utl_shap import shap_explain_target

features = resolve_features_to_model(config, df_enriched)
for target in targets:
    try:
        shap_values, feature_names, summary_fig, bar_fig = shap_explain_target(
            target=target,
            df_enriched=df_enriched,
            features_to_model=features,
            config=config,
            max_samples=1000,
        )
    except ImportError as exc:
        print(exc)
        print("Skipping SHAP. You can continue with Section 6.")
        break
    if summary_fig is None or bar_fig is None:
        print("SHAP skipped for", target)
        continue
    safe = _safe_target(target)
    file_io("save", "figure", FIGURES_DIR, f"shap_summary_{safe}.png", data=summary_fig)
    file_io("save", "figure", FIGURES_DIR, f"shap_importance_{safe}.png", data=bar_fig)


### From plots to a hypothesis

If monomer, additive, time, or ion size rank high *and* the direction matches what you know about interfacial polymerization, the model is at least consistent with chemistry. If the top feature is a paper ID or a column that never varies in the lab, the feature list is wrong.

Section 6 should only vary a **short** list of actionable knobs. If that list fights both SHAP and common sense, fix the YAML before trusting Ax.


## Section 6. Inverse design with Ax

**What you will do.** Treat ExtraTrees as a cheap stand-in for “make the membrane and measure it,” and let **Ax** search recipes that trade off permeability and rejection.

**Inverse design** means: we know the *performance we want* and search for *inputs* (the recipe). The optimizer **never** makes a real film. It only queries the model.

**Ax** is an open-source **Bayesian optimization** library. In plain language: it proposes the next recipe using a model of “where good results might be,” mixing **new** regions (exploration) with **known good** ones (exploitation).

**Analogy.** A flight simulator (the ExtraTrees model) vs the real aircraft (the lab). You try many cockpit settings in the simulator and keep a shortlist. You still have to fly the plane.

This tutorial uses **`optimization.mode: multiobjective`**: two goals at once, which produces a **Pareto set** (next subsection), not one winner.

```yaml
optimization:
  mode: multiobjective
  n_trials: 100          # more trials = slower, often a better front
  search_space: ...      # knobs Ax may change
  constant_parameters:   # knobs held fixed
```

**What to look for.** Ax must be installed (`pip install -e ".[opt]"`; first time is slow). You should see completed trials, printed recipes, then `inverse_design_summary.png`. On Colab: `/content/ML-MEM-Review/outputs/figures/`.

**Common pitfall.** Running this on **classification** labels, or without the `.pkl` files from Section 4. If you change `mode`, delete `models/ax_client.pkl` or keep `force_rebuild=True`.

**Try it yourself.** On a Pareto plot, what would a point that is high permeability *and* high rejection mean — and why might none of the trials sit there?


### Search space vs constants

**Search space:** knobs Ax is allowed to change — ranges (time, mass) or choices (monomer name). Listed under `optimization.search_space`.

**Constants:** everything the model still needs but you **refuse** to vary (`constant_parameters`), e.g. solvent or the test salt.

**Hard rule.** Searchable + constant columns together must match what ExtraTrees was trained on. If the fake experiment is missing a column, prediction is undefined.

Keep the space **small** so suggestions stay lab-plausible. SHAP (Section 5) can hint which knobs matter; the lab tells you which knobs you can actually turn.

**What to look for.** The next cell prints how many Ax parameters were built. Check that list against your protocol.


In [ ]:
if Client is None or RangeParameterConfig is None or ChoiceParameterConfig is None:
    import subprocess
    spec = f"{PROJECT_ROOT}[opt]"
    print("Ax not found; installing", spec)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig

from utl_Ax import (
    build_ax_search_space,
    get_optimization_recommendations,
    oracle_from_params,
    plot_ax_results,
    results_to_dataframe,
    run_ax_optimization,
)


In [ ]:
opt_cfg = config["optimization"]
search_space_parameters = build_ax_search_space(
    opt_cfg=opt_cfg,
    choice_param_cls=ChoiceParameterConfig,
    range_param_cls=RangeParameterConfig,
)
print("optimization.mode:", opt_cfg.get("mode"))
print("Search parameters from config.yaml:", len(search_space_parameters))
for item in search_space_parameters:
    print(" -", getattr(item, "name", item))


### The oracle (the model as a fake experiment)

Each Ax trial calls `oracle_from_params` — the **oracle** is just “ask the saved models”:

1. Merge the proposed knobs with `constant_parameters`.
2. Build a one-row table and run the **same** `feature_engineering` as Section 2.
3. Predict each KPI with the saved ExtraTrees pipelines.
4. Hand the numbers back to Ax.

You do not call the oracle yourself; `run_ax_optimization` does it every trial.

**Common pitfall.** Missing `models/*.pkl` (go back to Section 4) or a YAML name that is not a training feature.


### Two goals: a Pareto front, not one score

Permeability and rejection **compete**: open films pass more water *and* more salt. There is a **set** of good recipes, not a single winner.

A recipe is **dominated** if another recipe is better on **both** KPIs. The **Pareto frontier** is the set of recipes that are *not* dominated: you cannot improve one KPI without hurting the other.

This notebook uses `mode: multiobjective`. Objective `threshold` values in YAML are **soft constraints** on the *normalized* KPIs (scaled roughly 0–1 using `normalization_min` / `normalization_max`).

**What to look for.** The next cell prints `mode` and each objective’s direction and threshold. On the response maps, **Pareto points are blue triangles**; ordinary trials are red circles.

### Optional deeper dive

Set `mode: scalarized` and edit `scalarized_objective.equation` to collapse the two KPIs into **one weighted number** (example 05 does this). Faster to explain, but it hides the trade-off: the “best” recipe is only best *for those weights*.


In [ ]:
opt_cfg = config["optimization"]
print("mode:", opt_cfg.get("mode"))
for obj in opt_cfg.get("objectives") or []:
    print(
        f"  - {obj.get('name')}: {obj.get('direction')} "
        f"threshold={obj.get('threshold')} "
        f"-> {obj.get('model_target') or obj.get('original_name')}"
    )
eq = (opt_cfg.get("scalarized_objective") or {}).get("equation")
if opt_cfg.get("mode") == "scalarized":
    print("scalarized_objective.equation:", eq)


### The optimization loop

**What you will do.** Run `run_ax_optimization`: a few random (**Sobol**) start points, then Bayesian trials. `n_trials` in YAML is the budget. This cell can take **many minutes** if `n_trials` is large (the tutorial file uses 100).

**Sobol** is just a structured way to pick diverse random starts, better than throwing darts blindly.

**What to look for.** Logs ending with a completed-trial count. A checkpoint is saved under `models/` (`client_cache_file`). Re-running without `force_rebuild=True` may **reload** that file instead of searching again.

**Common pitfall.** No Ax in this kernel. Re-run bootstrap or `pip install -e ".[opt]"`.

Treat the result as a **ranked shortlist of experiments**, not a new membrane.


In [ ]:
client, results = run_ax_optimization(
    search_space_parameters=search_space_parameters,
    oracle_from_params=oracle_from_params,
    config=config,
    sobol_init=opt_cfg["sobol_init"],
    n_trials=opt_cfg["n_trials"],
    force_rebuild=True,
    random_seed=config.get("modeling", {}).get("random_state"),
    save_every_trials=opt_cfg["save_checkpoint_every"],
    client_path=os.path.join(MODEL_PATH, opt_cfg["client_cache_file"]),
)
print("Optimization completed:", len(results), "trials")


### Reading the suggested recipes

**What you will do.** Print recommendations (`get_optimization_recommendations`).

In **multiobjective** mode this is a **Pareto set**: several recipes, each a different permeability–rejection trade-off, under **this** search box and ExtraTrees surrogate — not a universally optimal membrane.

**What to look for.** Recipes a chemist would actually mix. If Ax proposes negative mass or a nonsense monomer pairing, the surrogate, bounds, or constants are wrong.

**Try it yourself.** Would you run the top recipe in the lab this week? What measurement would make you trust or reject the surrogate?


In [ ]:
recommendations = get_optimization_recommendations(
    client=client,
    optimization_mode=opt_cfg["mode"],
    top_k=5,
)


### Pictures of the search

**What you will do.** Save `inverse_design_summary.png` (how metrics moved over trials — the 3D “waterfall”) and **response maps** (smoothed surface of a KPI vs two knobs).

**Response map:** we interpolate the *trial* results onto a grid so you can see a landscape. With few trials the surface is a cartoon, not a map of nature. **Contour** = 2D view; **surface** = 3D view of the same interpolation.

Blue triangles (if present) = Pareto set. Red circles = all trials.

**What to look for.** Whether scores improved after the random start. If a map is skipped (too few distinct points), the summary figure is still enough.

Files: the figures directory from Section 0.


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
from utl_fileIO import file_io

trial_df = results_to_dataframe(results)
viz = plot_ax_results(trial_df=trial_df, config=config, client=client)

summary_figure = (
    viz.get("summary", {}).get("figure") if isinstance(viz.get("summary"), dict) else None
)
if summary_figure is None and viz.get("figures"):
    summary_figure = viz["figures"].get("summary_figure")
if summary_figure is not None:
    file_io("save", "figure", FIGURES_DIR, "inverse_design_summary.png", data=summary_figure)

for index, response_map in enumerate(viz.get("response_maps", []), start=1):
    if response_map.get("contour_figure") is not None:
        file_io(
            "save", "figure", FIGURES_DIR,
            f"response_map_{index}_contour.png",
            data=response_map["contour_figure"],
        )
    if response_map.get("surface_figure") is not None:
        file_io(
            "save", "figure", FIGURES_DIR,
            f"response_map_{index}_surface.png",
            data=response_map["surface_figure"],
        )

plt.close("all")
for name in [
    "inverse_design_summary.png",
    "response_map_1_contour.png",
    "response_map_1_surface.png",
    "response_map_2_contour.png",
    "response_map_2_surface.png",
]:
    path = Path(FIGURES_DIR) / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))


## Section 7. Recap, troubleshooting, next steps

You completed one loop: **table → extra chemistry columns → ExtraTrees → SHAP → Ax shortlist**.

### Key concepts

- **Features** vs **targets**; **train** vs **test**; **correlation** is not **causation**.
- The model is a **surrogate**. SHAP explains the surrogate. Ax **searches** the surrogate.
- Inverse design here is for **continuous** KPIs. Tutorial YAML uses **multiobjective (Pareto)**; example 05 uses a **scalarized** (weighted) score as a shorter demo.

### Troubleshooting

| Problem | What to try |
|---------|-------------|
| `config.yaml` / `utl_*` not found | Colab bootstrap, then Section 0, in order. |
| `No module named shap` | `pip install -e ".[shap]"` in **this** kernel; or skip to Section 6. |
| Ax / optimize error | `pip install -e ".[opt]"`; confirm Section 4 wrote `.pkl` files. |
| No Pareto triangles | `optimization.mode` must be `multiobjective`; delete `models/ax_client.pkl` and re-run with `force_rebuild=True`. |
| Old figures / old model | `force_retrain=True` / `force_rebuild=True`, or delete `models/`. |
| Colab clone looks old | Re-run bootstrap after a GitHub push. |

### Practice (pick one)

1. Change `data.targets` to another continuous column and re-run Sections 4–5.
2. Set `modeling.model_selection: randomforest` and compare test RMSE.
3. Set `optimization.mode: scalarized` and compare that single winner to the Pareto set.

None of these replace making the membrane. They help you choose **which** membrane to make next.
